# Solving a Cubic Equation

Cardano's method begins by translating a monic cubic to the
depressed form

$$
y^3+py+q=0.
$$

If $y=(s_1+s_2)/3$, then $s_1^3$ and $s_2^3$ are the roots of

$$
t^2+27qt-27p^3=0.
$$

The three choices are obtained with the cube root of unity
$\omega=(-1+i\sqrt3)/2$.


## Cardano's construction in Egison

The first match clause handles the depressed cubic. The second
performs the translation $x=y-b/3$, and the last divides by the
leading coefficient. A small typed quadratic helper keeps the
notebook self-contained.


In [1]:
declare symbol x, a, b, c, d, p, q: MathValue

def quadraticRoots
  (a : MathValue)
  (b : MathValue)
  (c : MathValue)
  : (MathValue, MathValue) :=
  ( ((- b) + sqrt (b ^ 2 - 4 * a * c)) / (2 * a)
  , ((- b) - sqrt (b ^ 2 - 4 * a * c)) / (2 * a) )

def omega : MathValue := ((-1) + i * sqrt 3) / 2

def cubicFormula : MathValue -> MathValue -> (MathValue, MathValue, MathValue) := cF

def cF (f : MathValue) (x : MathValue) : (MathValue, MathValue, MathValue) :=
  match coefficients f x as list mathValue with
    | [$a_0, $a_1, $a_2, $a_3] -> cF' a_3 a_2 a_1 a_0

def cF'
  (a : MathValue)
  (b : MathValue)
  (c : MathValue)
  (d : MathValue)
  : (MathValue, MathValue, MathValue) :=
  match (a, b, c, d) as (mathValue, mathValue, mathValue, mathValue) with
    | (#1, #0, $p, $q) ->
      let (s1, s2) := (2)#(rt 3 $1, rt 3 $2)
                         (quadraticRoots 1 (27 * q) ((-27) * p ^ 3))
       in ( (s1 + s2) / 3
          , (omega ^ 2 * s1 + omega * s2) / 3
          , (omega * s1 + omega ^ 2 * s2) / 3 )
    | (#1, _, _, _) ->
      (3)#($1 - b / 3, $2 - b / 3, $3 - b / 3)
        (withSymbols [x, y]
          cF
            (substitute
              [(x, y - b / 3)]
              (x ^ 3 + b * x ^ 2 + c * x + d))
            y)
    | (_, _, _, _) -> cF' 1 (b / a) (c / a) (d / a)


## The depressed cubic

Keeping $p$ and $q$ symbolic makes Cardano's radicals visible in
the output. The three tuple entries differ only by powers of
$\omega$.


In [2]:
cF (x ^ 3 + p * x + q) x


$(\frac{1}{3} \sqrt[3]{\frac{3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q} + \frac{1}{3} \sqrt[3]{\frac{-3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q}, \frac{-1}{3} \sqrt[3]{\frac{3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q} w + \frac{1}{3} \sqrt[3]{\frac{-3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q} w + \frac{-1}{3} \sqrt[3]{\frac{3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q}, \frac{1}{3} \sqrt[3]{\frac{3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q} w + \frac{-1}{3} \sqrt[3]{\frac{-3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q} w + \frac{-1}{3} \sqrt[3]{\frac{-3}{2} \sqrt{4 p^{3} + 27 q^{2}} \sqrt{3} + \frac{-27}{2} q})$

## Translation in action

The next polynomial is written in factored form so that its roots
are known in advance. Egison expands it, extracts its coefficients,
depresses it, and reconstructs all three roots.


In [3]:
cF ((x - 1) * (x - 2) * (x - 3)) x


$(2, \frac{2}{3} \sqrt{3} i w + \frac{1}{3} \sqrt{3} i + 2, \frac{-2}{3} \sqrt{3} i w + \frac{-1}{3} \sqrt{3} i + 2)$

## General leading coefficient

The same function accepts a fully symbolic non-monic cubic
$ax^3+bx^2+cx+d$. Its first action is normalization by $a$.


In [4]:
cF (a * x ^ 3 + b * x ^ 2 + c * x + d) x


$(\frac{-1}{3} a^{-1} b + \frac{1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1} + \frac{1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{-3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1}, \frac{-1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1} w + \frac{1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{-3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1} w + \frac{-1}{3} a^{-1} b + \frac{-1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1}, \frac{1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1} w + \frac{-1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{-3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1} w + \frac{-1}{3} a^{-1} b + \frac{-1}{3} \sqrt[3]{-b^{3} + \frac{-27}{2} a^{2} d + \frac{9}{2} a b c + \frac{-3}{2} \sqrt{4 b^{3} d - b^{2} c^{2} + 27 a^{2} d^{2} + 4 a c^{3} - 18 a b c d} \sqrt{3} a} a^{-1})$

## Takeaway

Cardano's formula becomes a short executable derivation when
coefficient patterns, substitution, and root permutations are
first-class operations. The three algebraic branches remain
visible instead of being hidden behind a generic solver.
